# CPP：本地 CPU 小实验

从 CIFAR-10 训练集随机取 1000 张图像，用论文对应的 CLIP ViT-L/14 提取特征，再训练官方 CPP 网络 3 轮。第一轮用于初始化。保持 CPP 的 4096 隐藏维度、128 输出维度和官方损失设置，训练批量为 25。

这是小规模流程与耗时实验。记录同一批样本上 CLIP + K-means、训练前 CPP 和训练后 CPP 的结果。

环境准备：在项目根目录执行 `uv sync --group cpp`，选择项目 `.venv` 内核。官方 CPP commit：`cb39bdc65f346b4cb5950d14af2e393a5f6a3566`。本次使用 CPU PyTorch 2.5.1、4 个计算线程。

In [1]:
from pathlib import Path
import sys
import time

import clip
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from torch.utils.data import DataLoader, Subset, TensorDataset
from tqdm import tqdm

root = Path.cwd().parent
sys.path.insert(0, str(root / "vendor" / "CPP"))
from data.dataset import load_dataset
from model.CPP_model import CPPNet
from model.sink_distance import SinkhornDistance
from loss.loss_fn import MLCLoss, TotalCodingRate
from metrics.clustering import spectral_clustering_metrics, clustering_accuracy

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.set_num_threads(4)
device = torch.device("cpu")
print("PyTorch:", torch.__version__, "device:", device, "threads:", torch.get_num_threads())

PyTorch: 2.5.1+cpu device: cpu threads: 4


## 提取 1000 张图像的特征

样本按固定随机种子选择，保留索引。CLIP 权重和数据在首次运行时下载，随后可直接使用本地文件。特征提取以 4 张图像为一批，单独记录模型加载和推理时间。

In [2]:
data_dir = root / "data" / "cpp"
data_dir.mkdir(parents=True, exist_ok=True)
dataset = load_dataset("cifar10", train=True, path=str(data_dir))
sample_indices = np.random.default_rng(seed).choice(len(dataset), 1000, replace=False)
image_loader = DataLoader(Subset(dataset, sample_indices), batch_size=4, shuffle=False, num_workers=0)

started = time.perf_counter()
encoder, _ = clip.load("ViT-L/14", device=device, download_root=str(root / "models" / "clip"))
encoder.eval()
model_load_seconds = time.perf_counter() - started

features, labels = [], []
started = time.perf_counter()
with torch.inference_mode():
    for images, targets in tqdm(image_loader, desc="CLIP features"):
        features.append(encoder.encode_image(images).float())
        labels.append(targets)
features = torch.cat(features)
labels = torch.cat(labels)
extraction_seconds = time.perf_counter() - started
del encoder

torch.save({"features": features, "ys": labels, "sample_indices": sample_indices},
           data_dir / "cifar10_clip_vitl14_cpu1000.pt")
print("features:", tuple(features.shape), "labels:", tuple(labels.shape))
print(f"CLIP load: {model_load_seconds:.1f}s, feature extraction: {extraction_seconds:.1f}s")

Files already downloaded and verified


CLIP features:   0%|          | 0/250 [00:00<?, ?it/s]

CLIP features:   0%|          | 1/250 [00:03<12:36,  3.04s/it]

CLIP features:   1%|          | 2/250 [00:05<11:18,  2.73s/it]

CLIP features:   1%|          | 3/250 [00:08<11:06,  2.70s/it]

...

CLIP features:  99%|█████████▉| 248/250 [18:04<00:05,  2.90s/it]

CLIP features: 100%|█████████▉| 249/250 [18:07<00:02,  2.89s/it]

CLIP features: 100%|██████████| 250/250 [18:09<00:00,  2.89s/it]

CLIP features: 100%|██████████| 250/250 [18:09<00:00,  4.36s/it]

features: (1000, 768) labels: (1000,)
CLIP load: 7.7s, feature extraction: 1090.0s


## 训练与评分

使用官方网络、关联矩阵和损失函数。训练前后都对这 1000 个样本一起做谱聚类；真实标签只用于评分。官方谱聚类函数含多次 K-means，这里保存每次评分及预测，不按真实标签选最好的一次。

In [3]:
model = CPPNet(input_dim=768, hidden_dim=4096, z_dim=128)
sinkhorn = SinkhornDistance(0.175, max_iter=5)
warmup_loss = TotalCodingRate(eps=0.1)
cpp_loss = MLCLoss(eps=0.1, gamma=1.0)
loader = DataLoader(TensorDataset(features), batch_size=25, shuffle=True)
feature_optimizer = torch.optim.SGD(
    list(model.pre_feature.parameters()) + list(model.subspace.parameters()),
    lr=1e-4, momentum=0.9, weight_decay=1e-4)
cluster_optimizer = torch.optim.SGD(
    model.cluster.parameters(), lr=1e-4, momentum=0.9, weight_decay=5e-3)

scores, predictions = [], {}
baseline = KMeans(n_clusters=10, random_state=seed, n_init=10).fit_predict(features.numpy())
predictions["clip_kmeans"] = baseline
scores.append({"method": "clip_kmeans", "run": 0,
               "acc": clustering_accuracy(labels.numpy(), baseline),
               "nmi": normalized_mutual_info_score(labels.numpy(), baseline),
               "ari": adjusted_rand_score(labels.numpy(), baseline)})

def evaluate_cpp(method):
    model.eval()
    with torch.inference_mode():
        _, codes = model(features)
        membership = sinkhorn((codes @ codes.T).abs()[None])[0][0] * len(features)
    acc, nmi, _, _, predicted = spectral_clustering_metrics(
        membership.numpy(), 10, labels.numpy(), verbose=False)
    predictions[method] = np.stack(predicted)
    for run, (a, n, p) in enumerate(zip(acc, nmi, predicted)):
        scores.append({"method": method, "run": run, "acc": a, "nmi": n,
                       "ari": adjusted_rand_score(labels.numpy(), p)})

evaluate_cpp("cpp_before")
training_log = []
started = time.perf_counter()
for epoch in range(3):
    model.train()
    losses = []
    for (x,) in loader:
        z, codes = model(x)
        if epoch == 0:
            loss = warmup_loss(z)
        else:
            membership = sinkhorn((codes @ codes.T).abs()[None])[0][0] * len(x)
            loss, _ = cpp_loss(z, membership)
            loss = loss + 0.05 * 0.5 * membership.norm().square()
        feature_optimizer.zero_grad()
        cluster_optimizer.zero_grad()
        loss.backward()
        feature_optimizer.step()
        if epoch > 0:
            cluster_optimizer.step()
        losses.append(loss.item())
    if epoch == 0:
        model.cluster.load_state_dict(model.subspace.state_dict())
    training_log.append({"epoch": epoch + 1, "loss": float(np.mean(losses))})
    print(training_log[-1])
training_seconds = time.perf_counter() - started
evaluate_cpp("cpp_after")
print(f"CPP training: {training_seconds:.1f}s")

{'epoch': 1, 'loss': -47.058135986328125}


{'epoch': 2, 'loss': -21.69140214920044}


{'epoch': 3, 'loss': -21.596992921829223}


CPP training: 8.5s


## 保存小实验结果

特征、模型与逐样本预测留在本地缓存目录；小型评分表保存在 results，方便后续核对。

In [4]:
score_table = pd.DataFrame(scores)
display(score_table.groupby("method")[["acc", "nmi", "ari"]].mean())
score_table.to_csv(root / "results" / "cpp_cpu1000_scores.csv", index=False)
pd.DataFrame(training_log).to_csv(root / "results" / "cpp_cpu1000_training.csv", index=False)
timings = {"clip_load_seconds": model_load_seconds, "extraction_seconds": extraction_seconds,
           "training_seconds": training_seconds, "n_samples": 1000, "batch_size": 25, "epochs": 3}
pd.DataFrame([timings]).to_csv(root / "results" / "cpp_cpu1000_timings.csv", index=False)
np.savez(data_dir / "cifar10_cpp_cpu1000_predictions.npz",
         labels=labels.numpy(), sample_indices=sample_indices, **predictions)
torch.save(model.state_dict(), root / "models" / "cpp_cifar10_cpu1000.pt")
print(timings)

,acc,nmi,ari
method,,,
clip_kmeans,0.8380,0.834052,0.783077
cpp_after,0.7275,0.672071,0.529755
cpp_before,0.7451,0.677462,0.610781


{'clip_load_seconds': 7.6544112999999925, 'extraction_seconds': 1089.9640262999983, 'training_seconds': 8.460713500000566, 'n_samples': 1000, 'batch_size': 25, 'epochs': 3}


## 本次运行结果

本次使用 1,000 张 CIFAR-10 图像，CPU 计算线程数为 4，训练批量为 25，训练 3 轮。CLIP 模型加载耗时 **7.7 秒**，特征提取耗时 **1,090.0 秒（约 18.2 分钟）**，CPP 训练耗时 **8.5 秒**。

| 方法 | ACC | NMI | ARI |
| --- | ---: | ---: | ---: |
| CLIP + K-means | 0.838 | 0.834 | 0.783 |
| CPP，训练前 | 0.745 | 0.677 | 0.611 |
| CPP，训练后 | 0.728 | 0.672 | 0.530 |

CPP 的指标取官方谱聚类函数 10 次输出的均值，所有预测均保存在本地。扩大样本数后，此次小批量、3 轮训练仍没有提高分数；这不能代替论文完整数据量和批量设置下的实验结果。